### Generate Oxford

# MiniGPT4

In [127]:
from string import punctuation

import pandas as pd
from scipy.special import huber
from sklearn.model_selection import train_test_split
import re
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
# from albumentations.core.transforms_interface import DualTransform, BasicTransform
# from transformers import BertTokenizer, AutoTokenizer, RobertaTokenizer,BertForMaskedLM, RobertaForMaskedLM, AutoModelForMaskedLM
# import albumentations
import torch
import re
import string


# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Minigpt4_Oxford.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# dirPath = '../Data/Instagram/CaptionID_sonicdrivein.csv'
# data = pd.read_csv(dirPath)

original = pd.read_csv('../Data/Instagram/CaptionID_mcdonalds_switzerland.csv')
generate = pd.read_csv('../Data/Instagram/Generate_mcdonalds_switzerland.csv')
original['caption'] = original['caption'].str.lower()
generate['caption'] = generate['caption'].str.lower()


In [128]:
image_id_counts = generate['image_id'].value_counts()
print('image_id_counts: ', image_id_counts.shape)
original['text_len'] = original['caption'].apply(lambda x: len(x.split()))
original['gen_count'] = original['image_id'].apply(lambda x: image_id_counts[x] if x in image_id_counts else 0)
original

image_id_counts:  (1125,)


,caption,image_id,funny_score,caption_id,text_len,gen_count
0,how cool is that?!😍 thanks gegette_and_co for ...,mcdonalds_switzerland_0,0.0,caption1,21,289
1,"for me, it's a deal breaker.",mcdonalds_switzerland_1,1.0,caption2,6,104
2,sweet tooth alert! please give a warm welcome ...,mcdonalds_switzerland_2,0.0,caption3,14,170
3,pov: you have a free menu with your loyalty po...,mcdonalds_switzerland_3,1.0,caption4,14,258
4,which shape do you prefer? mine is the bell 🔔\...,mcdonalds_switzerland_4,0.0,caption5,21,254
...,...,...,...,...,...,...
1120,the chips! #theprimeburger,mcdonalds_switzerland_2081,0.0,caption1125,3,24
1121,#theprimeburger #ingredients,mcdonalds_switzerland_2082,0.0,caption1126,2,10
1122,the delicious #theprimeburger,mcdonalds_switzerland_2084,0.0,caption1127,3,10
1123,#theprimeburger #new @reneschudel,mcdonalds_switzerland_2085,0.0,caption1128,3,35


In [129]:
image_id_counts = generate['image_id'].value_counts()
valid_image_ids = image_id_counts[(image_id_counts >= 200)].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = generate[generate['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
images = set(filtered_data['image_id'].unique())

shape of valid_image_ids:  (477,)
shape of filtered_data:  (163994, 3)


In [132]:
original = original[original['gen_count'] >= 100]
temp = original[original['text_len'] >= 12]
temp_img = set(temp['image_id'].unique())
print(len(temp_img), len(images))
print('intersection: ', len(images.intersection(temp_img)))
print('difference: ', len(images.difference(temp_img)))
print('difference: ', len(temp_img.difference(images)))

621 477
intersection:  466
difference:  11
difference:  155


In [9]:
temp = temp.sort_values(by=['funny_score'], ascending=[False])
temp[:100]

,caption,image_id,funny_score,caption_id,text_len,gen_count
827,we asked you for your favorite songs & artists...,sonicdrivein_831,1.0,caption828,25,438
306,no judgements at the drive-in. 🤣 watch the ful...,sonicdrivein_306,1.0,caption307,20,171
86,2024 solar eclipse 🤝 our new blackout slush fl...,sonicdrivein_86,1.0,caption87,19,160
307,the best car feature ever? no car seats. watch...,sonicdrivein_307,1.0,caption308,20,348
1454,when you look down and realize you ate all you...,sonicdrivein_1462,1.0,caption1458,17,234
...,...,...,...,...,...,...
475,"this season, give the gift of a $25 sonic gift...",sonicdrivein_479,0.0,caption476,26,397
478,when the food isn’t the only thing that’s comi...,sonicdrivein_482,0.0,caption479,18,242
480,treat your friends who are on the nice list th...,sonicdrivein_484,0.0,caption481,17,256
481,while we’re all in the queue for tickets… what...,sonicdrivein_485,0.0,caption482,18,276


In [ ]:
# ['text_len'] >= 24]
# 301 900
# intersection:  300
# difference:  600
# difference:  1

In [159]:
filtered_data['text_len'] = filtered_data['caption'].apply(lambda x: len(x.split()))
filtered_data.describe()

C:\Users\TonyLab\AppData\Local\Temp\ipykernel_6272\4250894817.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['text_len'] = filtered_data['caption'].apply(lambda x: len(x.split()))


,funny_score,text_len
count,334806.000000,334806.000000
mean,0.073222,39.927486
std,0.257638,49.493781
min,0.000000,9.000000
25%,0.000000,20.000000
50%,0.000000,24.000000
75%,0.000000,48.000000
max,1.000000,384.000000


In [163]:

image_id_counts = generate['image_id'].value_counts()
valid_image_ids = image_id_counts[(image_id_counts >= 100) & (image_id_counts < 200)].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = generate[generate['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
temp = filtered_data.merge(original, on='image_id', how='inner')
temp.describe()

shape of valid_image_ids:  (622,)
shape of filtered_data:  (91423, 3)


,funny_score_x,funny_score_y,text_len,aug_count
count,91423.000000,91423.000000,91423.000000,91423.000000
mean,0.239256,0.239256,12.118121,151.918368
std,0.420905,0.420905,3.742989,26.702788
min,0.000000,0.000000,5.000000,100.000000
25%,0.000000,0.000000,9.000000,131.000000
50%,0.000000,0.000000,11.000000,152.000000
75%,0.000000,0.000000,14.000000,173.000000
max,1.000000,1.000000,30.000000,199.000000


In [164]:
filtered_data['text_len'] = filtered_data['caption'].apply(lambda x: len(x.split()))
filtered_data.describe()

C:\Users\TonyLab\AppData\Local\Temp\ipykernel_6272\4250894817.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['text_len'] = filtered_data['caption'].apply(lambda x: len(x.split()))


,funny_score,text_len
count,91423.000000,91423.000000
mean,0.239256,12.936088
std,0.420905,3.945971
min,0.000000,5.000000
25%,0.000000,10.000000
50%,0.000000,12.000000
75%,0.000000,15.000000
max,1.000000,31.000000


In [215]:
split1 = original[original['gen_count'] >= 100]
split2 = generate.merge(split1, on='image_id', how='inner')
split2['text_len'] = split2['caption_x'].apply(lambda x: len(x.split()))
print("shape of splt1: ", split2.shape)

shape of splt1:  (426229, 8)


In [226]:
original = original[original['gen_count'] >= 100]
original = original[original['text_len'] >= 11]
data = data.merge(original, on='image_id', how='inner')
sss.shape

(1522, 6)


(1253, 6)

In [216]:
image_id_counts = split2['image_id'].value_counts()
print('image_id_counts: ', image_id_counts.shape)
valid_image_ids = image_id_counts[(image_id_counts >= 100) & (image_id_counts < 200)].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = split2[split2['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)

image_id_counts:  (1522,)
shape of valid_image_ids:  (622,)
shape of filtered_data:  (91423, 8)


In [219]:
image_id_counts = generate['image_id'].value_counts()
valid_image_ids_gen = image_id_counts[(image_id_counts >= 200)].index
print('image_id_counts: ', image_id_counts.shape)
print('valid_image_ids_gen: ', valid_image_ids_gen.shape)
filtered_data = generate[generate['image_id'].isin(valid_image_ids_gen)]
filtered_data['text_len'] = filtered_data['caption'].apply(lambda x: len(x.split()))
filtered_data

image_id_counts:  (2208,)
valid_image_ids_gen:  (900,)


C:\Users\TonyLab\AppData\Local\Temp\ipykernel_6272\1868607443.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['text_len'] = filtered_data['caption'].apply(lambda x: len(x.split()))


,caption,image_id,funny_score,text_len
1127,boss asked me to make something cool...what is...,sonicdrivein_12,0.0,12
1128,someone asked me to make something cool ... wh...,sonicdrivein_12,0.0,14
1129,you asked me to make something cool ... what i...,sonicdrivein_12,0.0,14
1130,he asked me to make something cool ... what is...,sonicdrivein_12,0.0,14
1131,they asked me to make something cool ... what ...,sonicdrivein_12,0.0,14
...,...,...,...,...
460259,"sonicland has moved to san antonio, tex. this ...",sonicdrivein_2237,0.0,14
460260,"sonicland has moved to san antonio, tex. this ...",sonicdrivein_2237,0.0,14
460261,"sonicland has moved to san antonio, tex. this ...",sonicdrivein_2237,0.0,14
460262,"sonicland has moved to san antonio, tex. this ...",sonicdrivein_2237,0.0,14


In [193]:
image_id_counts = split2['image_id'].value_counts()
valid_image_ids = image_id_counts[(image_id_counts >= 200)].index
for img in valid_image_ids:
    if img not in valid_image_ids_gen:
        print(img)
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = split2[split2['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)

shape of valid_image_ids:  (899,)
shape of filtered_data:  (334596, 8)


In [210]:
split1 = original[original['text_len'] >= 30]
image_id_counts = split1['image_id'].value_counts()

for img in image_id_counts.index:
    if img not in valid_image_ids_gen:
        print(img)
print('image_id_counts: ', image_id_counts.shape)

sonicdrivein_917
image_id_counts:  (179,)


In [23]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
def setofcategory(text):
    text = str(text)
    text = text.lower()
    # remove spaces
    text = text.strip()
    # split by ;
    text = text.split(';')
    # get set
    text = set(text)
    txt = ''
    for i in text:
        word = lemmatizer.lemmatize(i)
        if txt == '':
            txt = word
        else:
            txt += ';' + word
    return txt

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\TonyLab\AppData\Roaming\nltk_data...


In [25]:
data['emotion'] = data['emotion'].apply(lambda x: setofcategory(x))
data['sentiment'] = data['sentiment'].apply(lambda x: setofcategory(x))
data['humor'] = data['humor'].apply(lambda x: setofcategory(x))
data.to_csv(dirPath, index=False)

In [74]:
emotion_df = pd.DataFrame()
emotion_df['image_id'] = data['image_id']
# emotion_df['sentiment'] = data['sentiment']
emotion_df

,image_id
0,bokete_94229
1,imgflip_0
2,imgflip_1
3,imgflip_2
4,imgflip_3
...,...
429,imgflip_1472
430,imgflip_1508
431,imgflip_1535
432,imgflip_1774


In [76]:
all_emotion = set()
for i in range(len(emotion_df)):
    text = str(data['humor'][i])
    text = text.split(';')
    temp = set(text)
    all_emotion = all_emotion.union(temp)
len(all_emotion)

255

In [77]:
emotions = data['humor'].str.split(';').explode().unique()

# 建立 one-hot 編碼欄位
for emotion in emotions:
    emotion_df[emotion] = data['humor'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)

C:\Users\TonyLab\AppData\Local\Temp\ipykernel_2396\2476847226.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  emotion_df[emotion] = data['humor'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
C:\Users\TonyLab\AppData\Local\Temp\ipykernel_2396\2476847226.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  emotion_df[emotion] = data['humor'].apply(lambda x: 1 if emotion in str(x).split(';') else 0)
C:\Users\TonyLab\AppData\Local\Temp\ipykernel_2396\2476847226.py:5: PerformanceWarning: DataFrame is highly fragmented

In [78]:
print(emotion_df.shape)
emotion_df

(434, 256)


,image_id,irony,amusement,playful,sarcasm,none,exaggeration,mild,playfulness,innocence,...,privacy,seclusion,cumbersome,large,iron,satirical,crown,wearing,foreshadowing,carefree
0,bokete_94229,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,imgflip_0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,imgflip_1,1,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,imgflip_2,1,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,imgflip_3,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429,imgflip_1472,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
430,imgflip_1508,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
431,imgflip_1535,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
432,imgflip_1774,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [79]:
emotion_df.to_csv('../Data/Oxford_HIC/Minigpt4_Oxford_humor.csv', index=False)

In [83]:
emotion_df[emotion_df['image_id'] == 'imgflip_7']
print(emotion_df[emotion_df['image_id'] == 'imgflip_7'].shape)
# except image_id
print(emotion_df[emotion_df['image_id'] == 'imgflip_7'].iloc[:, 1:].shape)

(1, 256)
(1, 255)


In [84]:
emotion_df[emotion_df['image_id'] == 'imgflip_7'].iloc[:, 1:].values

array([[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [87]:
import torch        # to tensor
A = torch.tensor(emotion_df[emotion_df['image_id'] == 'imgflip_7'].iloc[:, 1:].values, dtype=torch.bfloat16)
# padding to 768
A = torch.nn.functional.pad(A, (0, 768 - A.shape[1]), 'constant', 0)
print(A.shape)

torch.Size([1, 255])


In [2]:
emotion_df= pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_emotion.csv')
sentiment = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_sentiment.csv')
humor = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_humor.csv')
print(emotion_df.shape, sentiment.shape, humor.shape)

(434, 402) (434, 419) (434, 256)


In [3]:
emotion_df['sum'] = emotion_df.iloc[:, 1:].sum(axis=1)
sentiment['sum'] = sentiment.iloc[:, 1:].sum(axis=1)
humor['sum'] = humor.iloc[:, 1:].sum(axis=1)
print(max(emotion_df['sum']), max(sentiment['sum']), max(humor['sum']))

15 12 14


In [ ]:
emotions_data = pd.read_csv('../../Oxford_HIC/Minigpt4_Oxford.csv')
for i in range(len(emotions_data)):
    tokens = torch.tensor(emotions_data.iloc[i, 1:].values, dtype=torch.bfloat16)

In [8]:
def addwhat(text, categorylist):
    if ':' in text:
        text = text.split(":")[0]
    elif '-' in text:
        text = text.split("-")[0]
    # remove punctuation and spaces
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    # check if only one word
    if '.' in text:
        text = text.split(".")[1]
        text = text.strip()
    if ' ' in text:
        return categorylist
    if categorylist != '':
        categorylist += ';' + text
    else:
        categorylist = text
    return categorylist

def categorize(text, emotion, sentiment, humor):
    split1 = text.split("\n\n")
    category = ''
    if emotion == 'nan':
        emotion = ''
    if sentiment == 'nan':
        sentiment = ''
    if humor == 'nan':
        humor = ''
    for i in range(len(split1)):
        if category == '':
            if 'emotion' in split1[i]:
                category = 'emotion'
                continue
            elif 'sentiment' in split1[i]:
                category = 'sentiment'
                continue
            elif 'humor' in split1[i]:
                category = 'humor'
                continue
        else:
            if '*' in split1[i]:
                split2 = split1[i].split("\n")
                for j in range(len(split2)):
                    if '*' in split1[i]:
                        if category == 'emotion':
                            emotion = addwhat(split2[j], emotion)
                        elif category == 'sentiment':
                            sentiment = addwhat(split2[j], sentiment)
                        elif category == 'humor':
                            humor = addwhat(split2[j], humor)
                category = ''
    return emotion, sentiment, humor


In [9]:
data['emotion'], data['sentiment'], data['humor']  = zip(*data.apply(lambda x: categorize(x['chat'], str(x['emotion']), str(x['sentiment']), str(x['humor'])), axis=1))

In [10]:
# if emotion, sentiment, humor are all filled done = 'O'
# if emotion, sentiment, humor are all empty done = 'X'
# if emotion, sentiment, humor are not all empty done = 'P'
def checkdone(emotion, sentiment, humor, done):
    if emotion != '' and sentiment != '' and humor != '':
        return 'O'
    else:
        return done

data['done'] = data.apply(lambda x: checkdone(x['emotion'], x['sentiment'], x['humor'], x['done']), axis=1)
data

,image_id,funny_score,chat,done,emotion,sentiment,humor
0,sonicdrivein_0,0.0,the image is of a large room with a blue floor...,O,relaxed;curious,neutral;calm,none
1,sonicdrivein_2,0.0,"the image shows a woman sitting on a couch, we...",done,,,
2,sonicdrivein_8,0.0,the image shows a group of people sitting in a...,O,happiness;joy;contentment;relaxation,positive;cheerful;lighthearted,playful;whimsical;silly
3,sonicdrivein_11,0.0,the image shows two people standing in front o...,done,,,
4,sonicdrivein_12,0.0,the image shows a person wearing blue shirt ho...,done,,,
...,...,...,...,...,...,...,...
1517,sonicdrivein_2235,0.0,this image shows a group of people standing on...,X,,,
1518,sonicdrivein_2236,0.0,this is an image of a man balancing on one leg...,X,,,
1519,sonicdrivein_2237,0.0,the image shows a plastic card with the batman...,O,happy;excited;creative,positive;funny,silly;quirky;playful
1520,sonicdrivein_2238,0.0,the image shows a group of people in rollerbla...,X,,,


In [11]:
data.to_csv('../Data/Instagram/Minigpt4_sonicdrivein.csv', index=False)

In [14]:
sonic

,image_id,emotion,sentiment,humor
0,sonicdrivein_0,modern;peaceful;futuristic;curious;relaxed,calm;enjoying;neutral;socialize,cursive;none;playful
1,sonicdrivein_2,happiness;contentment;relaxation,connection;intimacy;romance,playfulness;whimsy;irony
2,sonicdrivein_8,content;happy;ease;happiness;joy;positive;cont...,lighthearted;cheerful;none;positive,silly;none;playful;whimsical
3,sonicdrivein_11,engaged;focused;calm,none,none
4,sonicdrivein_12,surprise;wonder;curiosity,playfulness;joy;relaxation,whimsy;irony
...,...,...,...,...
1517,sonicdrivein_2235,joyful;excitement,success;achievement,confident;large;happy;pose
1518,sonicdrivein_2236,playfulness;balanced;joy;interacting;gathering...,friendliness;warmth;dexterity;joy;admiration;c...,comical;facial-expression
1519,sonicdrivein_2237,happy;creative;excited,positive;funny,silly;playful;quirky
1520,sonicdrivein_2238,joyfulness;excitement,unity;support,playfulness;energy


In [25]:
oxford = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford.csv')
mcdonald = pd.read_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland.csv')
sonic = pd.read_csv('../Data/Instagram/Minigpt4_sonicdrivein.csv')
oxford['emotion'] = ''
oxford['sentiment'] = ''
oxford['humor'] = ''
mcdonald['emotion'] = ''
mcdonald['sentiment'] = ''
mcdonald['humor'] = ''
sonic['emotion'] = ''
sonic['sentiment'] = ''
sonic['humor'] = ''


In [34]:
def getText(data):
    text = ''
    for column in data.index:
        if column == 'image_id':
            continue
        if data[column] == 1:
            if text == '':
                text = column
            else:
                text += ';' + column
    if text == '':
        text = 'none'
    return text

In [37]:
for category in ['emotion', 'sentiment', 'humor']:
    oxford_data = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford_' + category + '_filter.csv')
    mcdonald_data = pd.read_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland_' + category + '_filter.csv')
    sonic_data = pd.read_csv('../Data/Instagram/Minigpt4_sonicdrivein_' + category + '_filter.csv')

    oxford[category] = oxford_data.apply(lambda x: getText(x), axis=1)
    mcdonald[category] = mcdonald_data.apply(lambda x: getText(x), axis=1)
    sonic[category] = sonic_data.apply(lambda x: getText(x), axis=1)

In [40]:
oxford.to_csv('../Data/Oxford_HIC/Minigpt4_Oxford_filter.csv', index=False)
mcdonald.to_csv('../Data/Instagram/Minigpt4_mcdonalds_switzerland_filter.csv', index=False)
sonic.to_csv('../Data/Instagram/Minigpt4_sonicdrivein_filter.csv', index=False)

# ROUGE


In [1]:
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch
import torch.nn.functional as nnf
from typing import Tuple, List, Union, Optional
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    AdamW,
    get_linear_schedule_with_warmup,
)
from transformers import AutoConfig, AutoTokenizer, Gemma2ForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
import PIL.Image
from typing import Tuple, Optional, Union, Any
import os
import pickle
import sys
from typing import Tuple, Optional, Union, Any
import numpy as np
import pandas as pd
# from peft import LoraConfig, TaskType, get_peft_model
from nltk.translate.bleu_score import sentence_bleu
import gc
from tqdm import tqdm
from torch import Tensor
from nltk.translate.bleu_score import sentence_bleu
import gc
import loralib as lora
class OxfordDataset(torch.utils.data.Dataset):
    def __len__(self) -> int:
        return len(self.captions_tokens)

    def pad_tokens(self, item: int):
        tokens = self.captions_tokens[item].cpu()
        padding = self.max_seq_len - tokens.shape[0]
        if padding > 0:
            tokens = torch.cat((tokens, torch.zeros(padding, dtype=torch.int64) - 1))
            self.captions_tokens[item] = tokens
        elif padding < 0:
            tokens = tokens[:self.max_seq_len]
            self.captions_tokens[item] = tokens
        mask = tokens.ge(0)  # mask is zero where we out of sequence
        tokens[~mask] = 0
        mask = mask.float()
        mask = torch.cat((torch.ones(self.prefix_length), mask), dim=0)  # adding prefix mask
        return tokens, mask

    def __getitem__(self, item: int) -> tuple[Tensor, Tensor, Any, int]:
        tokens, mask = self.pad_tokens(item)
        if self.dataFrom == 'Oxford':
            prefix = torch.load('../../Oxford_HIC/ImageData/' + self.image_ids[item] + '.pt', weights_only=False)
        else:
            prefix = torch.load('../../Instagram/ImageData/'+ self.dataFrom +'/' + self.image_ids[item] + '.pt', weights_only=False)
        if self.normalize_prefix:
            prefix = prefix.float()
            prefix = prefix / prefix.norm(2, -1)
        return tokens, mask, prefix

    def __init__(self, data_path: str, prefix_length: int, gpt2_type: str = "gpt2", normalize_prefix=False, model=None,
                 batch_size=30, bleu_threshold=0.4, dataFrom='Oxford'):
        self.data_path = data_path
        self.dataFrom = dataFrom
        self.bleu = False
        # self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
        # self.tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
        self.tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
        # self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
        # self.tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
        # self.tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-2.7B")
        self.prefix_length = prefix_length
        self.normalize_prefix = normalize_prefix
        with open(data_path, 'rb') as f:
            all_data = pickle.load(f)
        sys.stdout.flush()
        self.prefixes = all_data["clip_embedding"]
        captions_raw = all_data["captions"]
        self.image_ids = [caption["image_id"] for caption in captions_raw]
        self.captions = [caption['caption'] for caption in captions_raw]
        if os.path.isfile(f"{data_path[:-4]}_bleu_all.pkl"):
            del all_data
            gc.collect()
            torch.cuda.empty_cache()
            with open(f"{data_path[:-4]}_bleu_all.pkl", 'rb') as f:
                self.captions_tokens, self.caption2embedding, self.max_seq_len = pickle.load(f)
            all_len = torch.tensor([len(self.captions_tokens[i]) for i in range(len(self))]).float()
            self.max_seq_len = min(int(all_len.mean() + all_len.std() * 10), int(all_len.max()))
        else:
            self.captions_tokens = []
            self.caption2embedding = []
            max_seq_len = 0
            for caption in captions_raw:
                self.captions_tokens.append(
                    torch.tensor(self.tokenizer.encode(caption['caption'], max_length=64, truncation=True),
                                 dtype=torch.int64))
                self.caption2embedding.append(caption["clip_embedding"])
                max_seq_len = max(max_seq_len, self.captions_tokens[-1].shape[0])
            self.max_seq_len = max_seq_len
            all_len = torch.tensor([len(self.captions_tokens[i]) for i in range(len(self))]).float()
            self.max_seq_len = min(int(all_len.mean() + all_len.std() * 10), int(all_len.max()))
            with open(f"{self.data_path[:-4]}_bleu_all.pkl", 'wb') as f:
                pickle.dump([self.captions_tokens, self.caption2embedding, self.max_seq_len], f)
        print(f"Train Data size: {len(self.captions_tokens)}")

class ClipCocoDataset(Dataset):

    def __len__(self) -> int:
        return len(self.captions_tokens)

    def pad_tokens(self, item: int):
        tokens = self.captions_tokens[item]
        padding = self.max_seq_len - tokens.shape[0]
        if padding > 0:
            tokens = torch.cat((tokens, torch.zeros(padding, dtype=torch.int64) - 1))
            self.captions_tokens[item] = tokens
        elif padding < 0:
            tokens = tokens[:self.max_seq_len]
            self.captions_tokens[item] = tokens
        mask = tokens.ge(0)  # mask is zero where we out of sequence
        tokens[~mask] = 0
        mask = mask.float()
        mask = torch.cat((torch.ones(self.prefix_length), mask), dim=0)  # adding prefix mask
        return tokens, mask

    def __getitem__(self, item: int) -> Tuple[torch.Tensor, ...]:
        tokens, mask = self.pad_tokens(item)
        prefix = self.prefixes[self.caption2embedding[item]]
        if self.normalize_prefix:
            prefix = prefix.float()
            prefix = prefix / prefix.norm(2, -1)
        return tokens, mask, prefix

    def __init__(self, data_path: str, prefix_length: int, gpt2_type: str = "gpt2",
                 normalize_prefix=False):
        # self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
        # self.tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
        self.tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
        self.prefix_length = prefix_length
        self.normalize_prefix = normalize_prefix
        with open(data_path, 'rb') as f:
            all_data = pickle.load(f)
        print("Data size is %0d" % len(all_data["clip_embedding"]))
        sys.stdout.flush()
        self.prefixes = all_data["clip_embedding"]
        captions_raw = all_data["captions"]
        self.image_ids = [caption["image_id"] for caption in captions_raw]
        self.captions = [caption['caption'] for caption in captions_raw]
        if os.path.isfile(f"{data_path[:-4]}_tokens.pkl"):
            with open(f"{data_path[:-4]}_tokens.pkl", 'rb') as f:
                self.captions_tokens, self.caption2embedding, self.max_seq_len = pickle.load(f)
        else:
            self.captions_tokens = []
            self.caption2embedding = []
            max_seq_len = 0
            for caption in captions_raw:
                self.captions_tokens.append(torch.tensor(self.tokenizer.encode(caption['caption']), dtype=torch.int64))
                self.caption2embedding.append(caption["clip_embedding"])
                max_seq_len = max(max_seq_len, self.captions_tokens[-1].shape[0])
            # self.max_seq_len = max_seq_len
            with open(f"{data_path[:-4]}_tokens.pkl", 'wb') as f:
                pickle.dump([self.captions_tokens, self.caption2embedding, max_seq_len], f)
        all_len = torch.tensor([len(self.captions_tokens[i]) for i in range(len(self))]).float()
        self.max_seq_len = min(int(all_len.mean() + all_len.std() * 10), int(all_len.max()))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# trainData = '../Data/Oxford_HIC/parse/oxford_lower_800up_only800_all_ViT-B_32_train.pkl'
# testData = '../Data/Oxford_HIC/parse/oxford_lower_800up_only800_rest_300up_top300_ViT-B_32_test.pkl'
trainData = '../Data/Instagram/parse/300up_only300_all_sonicdrivein_ViT-B_32_train.pkl'
testData = '../Data/Instagram/parse/300up_only300_rest_200up_top200_sonicdrivein_ViT-B_32_test.pkl'
prefix_length = 64
normalize_prefix = False
train_dataform = "sonicdrivein"
test_dataform = "sonicdrivein"
trainDataset = OxfordDataset(trainData, prefix_length, normalize_prefix=normalize_prefix, dataFrom = train_dataform)
testDataset = OxfordDataset(testData, prefix_length, normalize_prefix=normalize_prefix, dataFrom = test_dataform)

if train_dataform:
    ##################### oxford_300k #####################
    # train_image = ['imgflip_34', 'bokete_3820', 'imgflip_0','imgflip_8', 'imgflip_15', 'imgflip_19', 'bokete_104530','imgflip_730', 'imgflip_130', 'imgflip_677']
    # train_text = ['You finish doing something at your friends house and look at your phone; 7 missed calls from your mom; 7 missed calls from your mom'
    #               ,'I\'m in my 50s!'
    #               ,'School; Memes'
    #               ,'image tagged in memes,one does not simply'
    #               ,'THAT; IS WHAT A GOOD MEME LOOKS LIKE'
    #               ,'NOT SURE IF PEOPLE ARE UPVOTING MEMES; OR USER NAMES'
    #               ,'It\'s a family night runaway.'
    #               ,'CHUCK IS THE GOOD TYPE OF SCUMBAG; CUZ HE ONLY ROASTS YOU FROM YOUR INSIDES'
    #               ,'SO YOUR TELLIN\' ME THAT SCHOOLS GOOD FOR YOU'
    #               ,'Y\'ALL GOT ANY MORE OF THEM; JOBS?']
    ##################### oxford_100k #####################
    # train_image = ['imgflip_34', 'bokete_3820', 'imgflip_0','imgflip_8', 'imgflip_15', 'imgflip_19', 'bokete_104530','imgflip_730', 'imgflip_130', 'imgflip_677']
    # train_text = ['You finish doing something at your friends house and look at your phone; 7 missed calls from your mom; 7 missed calls from your mom'
    #               ,'I\'m in my 50s!'
    #               ,'School; Memes'
    #               ,'image tagged in memes,one does not simply'
    #               ,'THAT; IS WHAT A GOOD MEME LOOKS LIKE'
    #               ,'NOT SURE IF PEOPLE ARE UPVOTING MEMES; OR USER NAMES'
    #               ,'It\'s a family night runaway.'
    #               ,'Chuck Norris doesn\'t go washroom; He goes washBOOM!'
    #               ,'SO YOUR TELLIN\' ME THAT SCHOOLS GOOD FOR YOU'
    #               ,'Y\'ALL GOT ANY MORE OF THEM; JOBS?']
    ##################### oxford_Top10_300k #####################
    # train_image = ['bokete_100136', 'bokete_100144', 'bokete_100174','bokete_100193', 'bokete_100268', 'bokete_100287', 'bokete_100295','bokete_10031', 'bokete_100498', 'bokete_24339']
    # train_text = ['The wax isn\'t dry yet.'
    #               ,'I\'m sorry to hear that.'
    #               ,'You didn\'t put that microphone in the register, did you?'
    #               ,'I\'m showing my brother the privilege of being the youngest.'
    #               ,'Are you ready to join us?'
    #               ,'He\'s cute, he\'s 100% capable of killing.'
    #               ,'The ground suddenly fell to the left.'
    #               ,'"Mama, there\'s something in the front mat!"'
    #               ,'"You don\'t have a dad?" "You don\'t have a mom?"'
    #               ,'This month, I\'ve only got this much to offer.']
    ##################### oxford_Top10_300k_mess #####################
    # train_image = ['bokete_100345', 'bokete_100360', 'bokete_100364','bokete_100193', 'bokete_100372', 'bokete_100432', 'bokete_100295','bokete_100453', 'bokete_100498', 'bokete_100459']
    # train_text = ['I had a dream about going to school, so I want to take a day off from school.'
    #               ,'I went to the woods for a jog, and there was a lot of spider webs.'
    #               ,'I\'d like to ask you a different color.'
    #               ,'I\'m showing my brother the privilege of being the youngest.'
    #               ,'Ah! You bumped into me in the morning!'
    #               ,'Are you sure it\'s your dad who left you when you were three?'
    #               ,'The ground suddenly fell to the left.'
    #               ,'In the first place, there\'s a problem with Snow White, who eats apples given to an old lady who looks so bad.'
    #               ,'It was at this time that they switched.'
    #               ,'Did you think it was corn?']
    ##################### oxford_Only1200_300k #####################
    # train_image = ['imgflip_0', 'imgflip_101', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_47', 'imgflip_504']
    # train_text = ['12 dollars; 11 dollars with 1 dollar shipping'
    #               ,'I HAD A GIRLFRIEND; AAAAAAND ITS GONE'
    #               ,'I POUR THE CEREAL AFTER I POUR THE MILK'
    #               ,'WAITING FOR MY PHONE TO GET  TO 100%'
    #               ,'You when you have over one test at school in a day'
    #               ,'IF SOMEONE WANTS TO KILL YOU; GO TO A LIVING ROOM'
    #               ,'you; eating 5 pounds of cheese; every day; your stomach'
    #               ,'Me:stands up to stretch my legs; The person who had been pushing my wheelchair for the last 26 years'
    #               ,'Me: Opens door for some fresh air; Everyone else in the submarine:'
    #               ,'THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS']
    ##################### oxford_Only100_300k #####################
    # train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504']
    # train_text = ['climb a mountain? pff, i have wings...'
    #               ,'go to a pizza buffet eat all the pizza'
    #               ,'12 dollars; 11 dollars with 1 dollar shipping'
    #               ,'I POUR MILK BEFORE CEREAL'
    #               ,'ME WAITING FOR MY INTERNET TO RECONNECT'
    #               ,'calling the teacher mom'
    #               ,'IF SOMEONE DIES IN THE LIVING ROOM... IS IT STILL CALLED THE LIVING ROOM?'
    #               ,'you; losing a few seconds of your life looking at this'
    #               ,'me: gets up and starts clapping because the chiefs won; the guy who has been pushing my wheelchair for 10 years'
    #               ,'THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS']
    ##################### oxford_only10 #####################
    # train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504']
    # train_text = ['i\'ll take it sure ur not chicken?',
    #               'avoid all the work',
    #               'due tomorrow; do tomorrow',
    #               'i want to speak to your manager',
    #               'school: *closes*; the kid who was in the bathroom:',
    #               'almost getting pass an extremely hard level but failing last minute',
    #               'you can\'t lose your mind; if you don\'t have one',
    #               'you; doing nothing; today; teacher',
    #               'students: *acting crazy*; teacher: pay attention! that kid named attention:',
    #               'they called me four eyes. i call them no eyes.']
    ####################### default #######################
    train_image = []
    train_text = []
    for i in range(len(trainDataset)):
        caption = trainDataset.captions[i]
        image_id = trainDataset.image_ids[i]
        if image_id not in train_image:
            print(f"Image ID: {image_id}, Caption: {caption}")
            train_image.append(image_id)
            train_text.append(caption)
            if len(train_image) == 10:
                break
    #######################################################
    tokens_list = []
    mask_list = []
    prefix_list = []
    train_gt = []
    train_caption = dict()
    train_image_id_list = []
    if train_dataform != "Oxford":
        load = pd.read_csv(f'../Data/Instagram/CaptionID_{train_dataform}.csv')
        load['caption'] = load['caption'].str.lower()
        train_text = []
        for i in range(len(train_image)):
            caption = load[load['image_id'] == train_image[i]]['caption'].values[0]
            train_text.append(caption)
        inside = []
        for i in range(len(trainDataset)):
            caption = trainDataset.captions[i]
            image_id = trainDataset.image_ids[i]
            if image_id in train_image and image_id not in train_image_id_list:
                train_caption[image_id] = []
                train_caption[image_id].append(caption)
                tokens, mask, prefix = trainDataset[i]
                tokens_list.append(tokens)
                mask_list.append(mask)
                prefix_list.append(prefix)
                train_gt.append(caption)
                train_image_id_list.append(image_id)
    else:
        for i in range(len(trainDataset)):
            caption = trainDataset.captions[i]
            image_id = trainDataset.image_ids[i]
            if image_id in train_image:
                if train_caption.get(image_id) is not None:
                    train_caption[image_id].append(caption)
                else:
                    train_caption[image_id] = []
                    train_caption[image_id].append(caption)
                if caption in train_text and image_id not in train_image_id_list:
                    tokens, mask, prefix = trainDataset[i]
                    tokens_list.append(tokens)
                    mask_list.append(mask)
                    prefix_list.append(prefix)
                    train_gt.append(caption)
                    train_image_id_list.append(image_id)
    train_tokens = torch.stack(tokens_list).to(device)
    train_mask = torch.stack(mask_list).to(device)
    train_prefix = torch.stack(prefix_list).to(device)
    print(train_tokens.shape, train_mask.shape, train_prefix.shape, len(train_image_id_list))

if test_dataform:
    ##################### oxford_300k #####################
    # test_image = ['imgflip_7', 'imgflip_32']
    # test_text = ['CHEESE; ME AT 3 AM; CHEESE; MY MOM WHO WAS WAITING; ME'
    #               ,'IS THIS A PIGEON?']
    ##################### oxford_100k #####################
    # test_image = ['imgflip_7', 'imgflip_32']
    # test_text = ['THE DOG FOOD; MY DOG; DOG FOOD; ME LOOKING AT HIM; MY DOG'
    #               ,'MATH; ME; IS THIS THE REASON I DROPPED OUT OF COLLEGE?']
    ##################### oxford_10 k #####################
    # test_image = ['bokete_111723', 'imgflip_57']
    # test_text = ['I\'m going to take care of you. I\'m going to take care of you. I\'m going to take care of you.'
    #               ,'WHAT\'S DONE IN THE DARK WILL ALWAYS COME OUT IN THE LIGHT; BUT THATS NONE OF MY BUSINESS']
    ##################### oxford_Top10_300k ###############
    # test_image = ['are-you-serious-face', 'bokete_24326']
    # test_text = ['you have windows 98 seriously?'
    #               ,'The answer is 10.']
    ##################### oxford_Top10_300k_mess ##########
    # test_image = ['imgflip_156', 'bokete_24326']
    # test_text = ['Friend: I just had a dream in which I married my crush! My dreams:'
    #               ,'The answer is 10.']
    ##################### oxford_Only1200_300k ############
    # test_image = ['imgflip_130', 'imgflip_659']
    # test_text = ['0 VIEWS 5 DISLIKES'
    #               ,'when the mobile game ad is so laggy that it crashes your game and you lose out on a reward:']
    ##################### oxford_Only100_300k #############
    # test_image = ['i-love-coloring-kid', 'imgflip_130']
    # test_text = ['she started writing notes !!'
    #               ,'WHEN YOUR FRIEND; DOSENT LIKE ROOT BEER']
    ##################### oxford_only10 ###################
    # test_image = ['bokete_100174', 'imgflip_834']
    # test_text = ['get out of my way. i\'ll do it.'
    #                 ,'me fully prepared for the test; question 1']
    ####################### default #######################
    test_image = []
    test_text = []
    for i in range(len(testDataset)):
        caption = testDataset.captions[i]
        image_id = testDataset.image_ids[i]
        if image_id not in test_image:
            print(f"Image ID: {image_id}, Caption: {caption}")
            test_image.append(image_id)
            test_text.append(caption)
            if len(test_image) == 10:
                break
    #######################################################
    tokens_list = []
    mask_list = []
    prefix_list = []
    test_gt = []
    test_caption = dict()
    test_image_id_list = []

    if test_dataform != "Oxford":
        load = pd.read_csv(f'../Data/Instagram/CaptionID_{test_dataform}.csv')
        load['caption'] = load['caption'].str.lower()
        test_text = []
        for i in range(len(test_image)):
            caption = load[load['image_id'] == test_image[i]]['caption'].values[0]
            test_text.append(caption)
        for i in range(len(testDataset)):
            caption = testDataset.captions[i]
            image_id = testDataset.image_ids[i]
            if image_id in test_image and image_id not in test_image_id_list:
                test_caption[image_id] = []
                test_caption[image_id].append(caption)
                tokens, mask, prefix = testDataset[i]
                tokens_list.append(tokens)
                mask_list.append(mask)
                prefix_list.append(prefix)
                test_gt.append(caption)
                test_image_id_list.append(image_id)
    else:
        for i in range(len(testDataset)):
            caption = testDataset.captions[i]
            image_id = testDataset.image_ids[i]
            if image_id in test_image:
                if test_caption.get(image_id) is not None:
                    test_caption[image_id].append(caption)
                else:
                    test_caption[image_id] = []
                    test_caption[image_id].append(caption)
                if caption in test_text and image_id not in test_image_id_list:
                    # print(f"Image ID: {image_id}, Caption: {caption}")
                    tokens, mask, prefix = testDataset[i]
                    tokens_list.append(tokens)
                    mask_list.append(mask)
                    prefix_list.append(prefix)
                    test_gt.append(caption)
                    test_image_id_list.append(image_id)

    test_tokens = torch.stack(tokens_list).to(device)
    test_mask = torch.stack(mask_list).to(device)
    test_prefix = torch.stack(prefix_list).to(device)
    print(test_tokens.shape, test_mask.shape, test_prefix.shape)

Train Data size: 134100
Train Data size: 90600
Image ID: sonicdrivein_1006, Caption: we got cheesecake blasts, with actual cheesecake pieces in it! 💞 choose between our classic cheesecake and strawberry cheesecake blasts, only at sonic.

at participating sonic® drive-ins for a limited time. mobile ordering available at select locations; hours may vary. tm & ©2021 america’s drive-in brand properties llc
Image ID: sonicdrivein_1009, Caption: we got a wacky deal headed your way! 🤪 buy a wacky pack kids meal online or in the sonic app, from may 24 - 30, and get a free small blast reward for your next visit! treats for the whole family! 👪

*see app for details
Image ID: sonicdrivein_1012, Caption: warm chili and cheese goodness, crispy fried onion crunch, and allllll of the southwest flavor we could pack on top of our 100% all beef patty. try a twisted texan cheeseburger or footlong coney today.

try one at 1/2 price when you order online or in the sonic app.
Image ID: sonicdrivein_1013, Ca

In [2]:
save_file = '20250216_oxford_lower_only800_base_sonicdrivein_only300_transformer_lora_p64_falcon_swin_tf8'
i = 2
result = pd.read_csv(f'./Model/{save_file}/oxford/{save_file}_test_{i :03d}.csv')
textList = result['text'].tolist()
test_generate_beam = textList[1:11]
train_generate_beam = textList[11:21]
test_generate2 = textList[24:34]
train_generate2 = textList[34:44]
test_train = textList[27:37]
train_train = textList[37:47]

In [3]:
test_groundtruth = dict()
train_groundtruth = dict()
test_generate_beam_dict = dict()
train_generate_beam_dict = dict()
test_generate2_dict = dict()
train_generate2_dict = dict()
test_train_dict = dict()
train_train_dict = dict()

for i in range(10):
    all_captions = []
    for j in range(len(test_caption[test_image[i]])):
        all_captions.append(dict(caption=test_caption[test_image[i]][j]))
    test_groundtruth[test_image[i]] = all_captions
    all_captions = []
    for j in range(len(train_caption[train_image[i]])):
        all_captions.append(dict(caption=train_caption[train_image[i]][j]))
    train_groundtruth[train_image[i]] = all_captions
    test_generate_beam_dict[test_image[i]] = [dict(caption=test_generate_beam[i])]
    train_generate_beam_dict[train_image[i]] = [dict(caption=train_generate_beam[i])]
    test_generate2_dict[test_image[i]] = [dict(caption=test_generate2[i])]
    train_generate2_dict[train_image[i]] = [dict(caption=train_generate2[i])]
    test_train_dict[test_image[i]] = [dict(caption=test_train[i])]
    train_train_dict[train_image[i]] = [dict(caption=train_train[i])]

In [4]:
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.meteor.meteor import Meteor
# from Citations.cococaption.pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
print('tokenization...')
tokenizer = PTBTokenizer()
gts  = tokenizer.tokenize(gts)
res = tokenizer.tokenize(res)

# =================================================
# Set up scorers
# =================================================
print('setting up scorers...')
scorers = [
    (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    (Meteor(),"METEOR"),
    (Rouge(), "ROUGE_L"),
    (Cider(), "CIDEr"),
    (Spice(), "SPICE")
]

# =================================================
# Compute scores
# =================================================
for scorer, method in scorers:
    print('computing %s score...'%(scorer.method()))
    score, scores = scorer.compute_score(gts, res)
    if type(method) == list:
        for sc, scs, m in zip(score, scores, method):
            self.setEval(sc, m)
            self.setImgToEvalImgs(scs, gts.keys(), m)
            print("%s: %0.3f"%(m, sc))
    else:
        self.setEval(score, method)
        self.setImgToEvalImgs(scores, gts.keys(), method)
        print("%s: %0.3f"%(method, score))

AttributeError: 'COCOEvalCap' object has no attribute 'params'

In [ ]:
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.meteor.meteor import Meteor
# from Citations.cococaption.pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
# from Citations.cococaption.pycocoevalcap.wmd.wmd import WMD


class Scorer():
    def __init__(self, cap, gt):
        self.evalImgs = []
        self.eval = {}
        self.imgToEval = {}
        tokenizer = PTBTokenizer()
        self.res  = tokenizer.tokenize(cap)
        self.gts = tokenizer.tokenize(gt)
        print('setting up scorers...')
        self.scorers = [
            (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
            (Meteor(), "METEOR"),
            (Rouge(), "ROUGE_L"),
            (Cider(), "CIDEr"),
            # (Spice(), "SPICE"), #需要圖片的annotation
            # (WMD(), "WMD")
        ]

    def compute_score(self):
        final_scores = {}
        print("==========")
        for scorer, method in self.scorers:
            print('computing %s score...'%(scorer.method()))
            score, scores = scorer.compute_score(self.gts, self.res)
            if type(method) == list:
                for sc, scs, m in zip(score, scores, method):
                    self.setEval(sc, m)
                    self.setImgToEvalImgs(scs, self.gts.keys(), m)
                    print("%s: %0.3f"%(m, sc))
            else:
                self.setEval(score, method)
                self.setImgToEvalImgs(scores, self.gts.keys(), method)
                print("%s: %0.3f"%(method, score))
    def setEval(self, score, method):
        self.eval[method] = score
    def setImgToEvalImgs(self, scores, imgIds, method):
        for imgId, score in zip(imgIds, scores):
            if not imgId in self.imgToEval:
                self.imgToEval[imgId] = {}
                self.imgToEval[imgId]["image_id"] = imgId
            self.imgToEval[imgId][method] = score

scorer = Scorer(test_generate_beam_dict, test_groundtruth)
scorer.compute_score()

setting up scorers...
computing Bleu score...
{'testlen': 0, 'reflen': 0, 'guess': [0, 0, 0, 0], 'correct': [0, 0, 0, 0]}
ratio: 1e-06
Bleu_1: 0.000
Bleu_2: 0.000
Bleu_3: 0.000
Bleu_4: 0.000
computing METEOR score...


In [23]:
import json
traintest = 'test'
now = 'generate_beam'
new_cap = []
for i in range(10):
    if traintest == 'train':
        if now == 'generate_beam':
            new_cap.append({'image_id': train_image[i], 'caption': train_generate_beam[i]})
        elif now == 'generate2':
            new_cap.append({'image_id': train_image[i], 'caption': train_generate2[i]})
        else:
            new_cap.append({'image_id': train_image[i], 'caption': train_train[i]})
    else:
        if now == 'generate_beam':
            new_cap.append({'image_id': test_image[i], 'caption': test_generate_beam[i]})
        elif now == 'generate2':
            new_cap.append({'image_id': test_image[i], 'caption': test_generate2[i]})
        else:
            new_cap.append({'image_id': test_image[i], 'caption': test_train[i]})

new_ref = {'images': [], 'annotations': []}
for k, refs in test_caption.items():
    new_ref['images'].append({'id': k})
    for ref in refs:
        new_ref['annotations'].append({'image_id': k, 'id': k, 'caption': ref})

with open(f'./Model/{save_file}/oxford/references_{traintest}_{now}.json', 'w') as fgts:
    json.dump(new_ref, fgts)
with open(f'./Model/{save_file}/oxford/captions_{traintest}_{now}.json', 'w') as fres:
    json.dump(new_cap, fres)

In [24]:
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap

annotation_file = f'./Model/{save_file}/oxford/references_{traintest}_{now}.json'
results_file = f'./Model/{save_file}/oxford/captions_{traintest}_{now}.json'

# create coco object and coco_result object
coco = COCO(annotation_file)
coco_result = coco.loadRes(results_file)

# create coco_eval object by taking coco and coco_result
coco_eval = COCOEvalCap(coco, coco_result)

# evaluate on a subset of images by setting
# coco_eval.params['image_id'] = coco_result.getImgIds()
# please remove this line when evaluating the full validation set
#coco_eval.params['image_id'] = coco_result.getImgIds()

# evaluate results
# SPICE will take a few minutes the first time, but speeds up due to caching
coco_eval.evaluate()

# print output evaluation scores
for metric, score in coco_eval.eval.items():
    print(f'{metric}: {score:.3f}')

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
tokenization...


FileNotFoundError: [WinError 2] 系統找不到指定的檔案。

# Oxford_HIC / Instagram Loading

In [ ]:
from string import punctuation

import pandas as pd
from scipy.cluster.hierarchy import single
from sklearn.model_selection import train_test_split
# create a dataframe with text and numbers
data = {
    'Text': ['Hello', 'World', 'Test', 'DataFrame'],
    'Numbers': [1, 2, 3, 4],
    'another':[52,47,88,77]
}

# Create a DataFrame
df = pd.DataFrame(data)
avgscore = pd.DataFrame()
for column in df.columns:
    if column == 'Text':
        avgscore[column] = ["Average"]
    elif pd.api.types.is_numeric_dtype(df[column]):
        avgscore[column] = [df[column].mean()]
    else:
        avgscore[column] = ["-"]
generate_beam_df = pd.concat([df, avgscore], axis=0)
generate_beam_df

In [ ]:
from string import punctuation

import pandas as pd
from scipy.cluster.hierarchy import single
from sklearn.model_selection import train_test_split

# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/Generate_sonicdrivein.csv'

data = pd.read_csv(dirPath)
data['caption'] = data['caption'].str.lower()

In [ ]:
dirPath = '../Data/Instagram/Generate_sonicdrivein.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
image_id_counts = data['image_id'].value_counts()
print(f'image_id_counts: ', len(image_id_counts))
######################################################################################################
valid_image_ids = image_id_counts[image_id_counts >= 100].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = data[data['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
train = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(100)
)
print("shape of train: ", train.shape)

valid_image_ids = image_id_counts[(image_id_counts >= 50) & (image_id_counts < 100)].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = data[data['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
test = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(50)
)
print("shape of test: ", test.shape)

# unique_image_ids = data['image_id'].unique()
# train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
# train = data[data['image_id'].isin(train_ids)]
# image_id_counts = train['image_id'].value_counts()
# print(f'train image_id_counts: ', len(image_id_counts))
# test = data[data['image_id'].isin(test_ids)]
# image_id_counts = test['image_id'].value_counts()
# print(f'test image_id_counts: ', len(image_id_counts))
# print(train.shape, test.shape)

# COCO Loading

In [ ]:
### only left 4090 has coco ###
import json
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_train2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
imageList = list()
captionList = list()
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    imageList.append(image_id)
    captionList.append(caption)
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_val2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    imageList.append(image_id)
    captionList.append(caption)

In [ ]:
import pandas as pd
data= pd.DataFrame()
data['image_id'] = imageList
data['caption'] = captionList
data

# Count in dictionary words and out dictionary words

In [ ]:
#compute the min, max, mean, and variance of the number of tokens in each caption
import nltk
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer
import torch
inDictSet = set()
outDictSet = set()
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")

def inDictionary(caption):
    caption = caption.lower()
    split = nltk.word_tokenize(caption)
    inCount = 0
    outCount = 0
    for word in split:
        if word in inDictSet:
            inCount += 1
        elif word in outDictSet:
            outCount += 1
        else:
            token = tokenizer(word, return_tensors="pt").input_ids
            if token.shape[1] == 1:
                inDictSet.add(word)
                inCount += 1
            else:
                outDictSet.add(word)
                outCount += 1
    print(f'inCount: {inCount}, outCount: {outCount}')
    return inCount, outCount

In [ ]:
data['inDictCount'], data['outDictCount'] = zip(*data['caption'].map(inDictionary))
print(data.shape)
data.describe()

get data similar to COCO (inDictCount = 10, inDictCount = 1)

In [ ]:
#keep data with outDictCount = 0
# new_data = data[data['outDictCount'] <= 3]
# print(new_data.shape)
new_data = data[data['inDictCount'] <= 60]
print(new_data.shape)
new_data.describe()

In [ ]:
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
print("shape of data: ", new_data.shape)
print()
image_id_counts = new_data['image_id'].value_counts()
######################################################################################################
valid_image_ids = image_id_counts[image_id_counts >= 7].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
train = (
    filtered_data.sort_values(by=['image_id','funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(7)
)
print("shape of train: ", train.shape)
train.to_csv('../Data/Instagram/Only7_simCOCO_mcdonalds_switzerland.csv', index=False)

keep data with outDictCount = 0

In [ ]:
data = data[data['outDictCount'] == 0]
print(data.shape)
data.to_csv('../Data/Oxford_HIC/NoOutDict_oxford_hic_data.csv', index=False)

funny_score top n caption

In [ ]:
top_captions = (
    data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(10)
)
top_captions.shape
top_captions.to_csv('../Data/Oxford_HIC/NoOutDictTop10_oxford_hic_data.csv', index=False)

# Generate data

In [1]:
from string import punctuation
import pandas as pd
from sklearn.model_selection import train_test_split

dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# dirPath = '../Data/Instagram/CaptionID_sonicdrivein.csv'

data = pd.read_csv(dirPath)
data['caption'] = data['caption'].str.lower()
data['fc_mean'] = data.groupby('image_id')['funny_score_y'].transform('mean')
data

,Unnamed: 0,image_id,caption,funny_score_x,caption_id,funny_score_y,fc_mean
0,0,bokete_0,my driver's license photo,0.0,caption1,0.0,1.100000
1,1,bokete_1,refugee relief.,0.0,caption2,0.0,0.421053
2,2,bokete_2,now! i think i stepped on a cat! what? really?...,0.0,caption3,0.0,0.357143
3,3,bokete_3,you wouldn't know i was reading a comic book.,0.0,caption4,0.0,1.096774
4,4,bokete_4,"oh no! i forgot my ・・・・ clothes!""",0.0,caption5,0.0,2.527273
...,...,...,...,...,...,...,...
3259948,3427273,british-high-school-honeybee,where you're supposed to be,0.0,caption3398077,0.0,0.000000
3259949,3427274,british-high-school-honeybee,on time,0.0,caption3398078,0.0,0.000000
3259950,3427275,british-high-school-honeybee,quiz tomorrow bee ready!!!,0.0,caption3398079,0.0,0.000000
3259951,3427276,british-high-school-honeybee,i'll die no matter what my fucking defenses bitch,0.0,caption3398080,0.0,0.000000


In [2]:
onlyone = (
    data.sort_values(by=['image_id', 'fc_mean'], ascending=[True, False])
    .groupby('image_id')
    .head(1)
)
onlyone = onlyone.sort_values(by=['fc_mean'], ascending=[False])
onlyone

,Unnamed: 0,image_id,caption,funny_score_x,caption_id,funny_score_y,fc_mean
1156030,1165412,bokete_91291,i don't care if i'm floating on my nipples!,0.000020,caption1165391,2.0,4442.600000
1476670,1490751,bokete_110977,let's give it a round of applause.,0.000031,caption1490717,3.0,3557.833333
1180491,1190074,bokete_93606,father rushed in to cheer you up from honolulu.,0.000000,caption1190054,0.0,3388.416667
1354510,1367986,bokete_103390,(laughter),0.000000,caption1367956,0.0,3255.866667
1282427,1293781,bokete_99998,"""can i do it? kaabako?"" ""huh!""",0.000031,caption1293751,3.0,3024.611111
...,...,...,...,...,...,...,...
3146433,3306703,advice-peeta,mother says she's too young to have a boyfrien...,0.000000,caption3277507,0.0,0.000000
3150535,3310985,afraid-yao-ming-trollface,damn internet you scary,0.000000,caption3281789,0.0,0.000000
1041936,1050521,bokete_80694,"he was dressed in white, and he was fed by the...",0.000000,caption1050499,0.0,0.000000
3119574,3278971,american-pride-eagle,land of the free* *some restrictions may apply,0.000000,caption3249775,0.0,0.000000


In [3]:
had = pd.read_csv('../Data/Oxford_HIC/Minigpt4_Oxford.csv')
had

,image_id,emotion,sentiment,humor
0,bokete_94229,seriousness;introspection,melancholy;reflection,amusement;irony
1,imgflip_0,joyful,energetic,playful
2,imgflip_1,sad;confused;happy,positive;negative;neutral,none;irony;sarcasm
3,imgflip_2,surprise;frustration;sadness;curiosity,playfulness;anticipation;melancholic,exaggeration;none;irony
4,imgflip_3,intensity;anger;violence;fear,heroism;villainy;gritty;dark,irony
...,...,...,...,...
429,imgflip_1472,surprise;shock;disbelief,awe;fear,none
430,imgflip_1508,dark;serious,disapproval;disappointment,humor;irony;juxtaposition
431,imgflip_1535,happy;relaxed,warm;pleasant;inviting;friendly,none
432,imgflip_1774,enjoyment;happiness;joy;amusement;comfortable;...,positive,exaggerate


In [4]:
print('onlyone: ', onlyone.shape)
print('had: ', had.shape)
onlyone_img = set(onlyone[:1000]['image_id'].tolist())
had_img = set(had['image_id'].tolist())
print('intersection: ', len(onlyone_img.intersection(had_img)))
print('difference: ', len(onlyone_img.difference(had_img)))
print('difference: ', len(had_img.difference(onlyone_img)))

onlyone:  (116649, 7)
had:  (434, 4)
intersection:  2
difference:  998
difference:  432


In [5]:
onlyone_img.union()
newDF = pd.DataFrame(list(onlyone_img.union(had_img)), columns=['image_id'])
newDF.shape

(1432, 1)

In [6]:
data = data.merge(newDF, on='image_id', how='inner', suffixes=('', '_y'))
image_id_counts = data['image_id'].value_counts()
print(len(image_id_counts))

1432


In [7]:
data['generated'] = 0

In [8]:
import pandas as pd
print("shape of data: ", data.shape)
print()
image_id_counts = data['image_id'].value_counts()
######################################################################################################
print(f'                  Oxford_HIC')
print(f'           Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 300]
# sum = len(x)
# print(f' {300} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 200]
# x = x[x < 300]
# sum += len(x)
# print(f'{200:4d} <= caption < {300:4d} --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 100]
# x = x[x < 200]
# sum += len(x)
# print(f'{100:4d} <= caption < {200:4d} --- {len(x):6d} --- {sum:6d}')
for i in range(10):
    x = image_id_counts[image_id_counts >= (10-1-i)*100]
    x = x[x < (10-i)*100]
    sum += len(x)
    print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


shape of data:  (1484182, 8)

                  Oxford_HIC
           Total image counts: 1432
 1000 <= caption        ---    203 ---    203
  900 <= caption < 1000 ---     10 ---    213
  800 <= caption <  900 ---     11 ---    224
  700 <= caption <  800 ---     19 ---    243
  600 <= caption <  700 ---     23 ---    266
  500 <= caption <  600 ---     35 ---    301
  400 <= caption <  500 ---     41 ---    342
  300 <= caption <  400 ---     92 ---    434
  200 <= caption <  300 ---      3 ---    437
  100 <= caption <  200 ---      4 ---    441
    0 <= caption <  100 ---    991 ---   1432


In [9]:
import pandas as pd
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
import albumentations
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

class NLPTransform(BasicTransform):
    """ Transform for nlp task."""

    @property
    def targets(self):
        return {"data": self.apply}

    def update_params(self, params, **kwargs):
        if hasattr(self, "interpolation"):
            params["interpolation"] = self.interpolation
        if hasattr(self, "fill_value"):
            params["fill_value"] = self.fill_value
        return params

    def get_sentences(self, text, lang='en'):
        return sent_tokenize(text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\TonyLab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\TonyLab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
from transformers import RobertaTokenizer,RobertaForMaskedLM
# from transformers import BertTokenizer, AutoTokenizer, RobertaTokenizer,BertForMaskedLM, RobertaForMaskedLM, AutoModelForMaskedLM
import albumentations
import torch
import re
import string

class LMmask(NLPTransform):

    def __init__(self, mask_num = 1, tokenizer_name='bert-base-uncased', model_name='bert-base-uncased'):
        self.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~’'
        self.mask_num = mask_num
        if tokenizer_name == 'BertTokenizer':
            #   BertTokenizer + BertForMaskedLM  ==> 'bert-base-uncased'
            self.tokenizer = BertTokenizer.from_pretrained(model_name)
            self.model = BertForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 103
        elif tokenizer_name == 'RobertaTokenizer':
            #   RobertaTokenizer + RobertaForMaskedLM  ==> 'FacebookAI/roberta-base'
            self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
            self.model = RobertaForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 50264
        elif tokenizer_name == 'AutoTokenizer':
            # AlbertConfig, BartConfig, BertConfig, BigBirdConfig, CamembertConfig, ConvBertConfig, Data2VecTextConfig, DebertaConfig, DebertaV2Config, DistilBertConfig, ElectraConfig, ErnieConfig, EsmConfig, FlaubertConfig, FNetConfig, FunnelConfig, IBertConfig, LayoutLMConfig, LongformerConfig, LukeConfig, MBartConfig, MegaConfig, MegatronBertConfig, MobileBertConfig, MPNetConfig, MraConfig, MvpConfig, NezhaConfig, NystromformerConfig, PerceiverConfig, QDQBertConfig, ReformerConfig, RemBertConfig, RobertaConfig, RobertaPreLayerNormConfig, RoCBertConfig, RoFormerConfig, SqueezeBertConfig, TapasConfig, Wav2Vec2Config, XLMConfig, XLMRobertaConfig, XLMRobertaXLConfig, XmodConfig, YosoConfig.
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)


            self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def apply(self, data, top_k=10):
        new_text = []
        split = nltk.word_tokenize(data)
        for idx, n in enumerate(split):
            words = ''
            not_enough = False
            for i in range(self.mask_num):
                if idx + i >= len(split):
                    not_enough = True
                    break
                if split[idx + i] in self.punctuation:
                    words = words + split[idx + i]
                else:
                    words = words + ' ' + split[idx + i]
            if not_enough:
                continue
            words = re.sub(r'\s+', ' ', words)
            words = words.strip()
            token = self.tokenizer.encode(words, return_tensors="pt")
            single = token.shape[1] == self.mask_num + 2
            if single:
                text = ''
                words = nltk.word_tokenize(words)
                for idx_w, word_split in enumerate(split):
                    if word_split in words and idx_w in range(idx, idx + self.mask_num):
                        words.remove(word_split)
                        text = text + ' ' + self.mask_token
                    elif word_split in self.punctuation:
                        text = text + word_split
                    else:
                        text = text + ' ' + word_split
                text = re.sub(r'\s+', ' ', text)
                text = text.strip()
                # DEFINE SENTENCE
                indices = self.tokenizer.encode(text, add_special_tokens=True, return_tensors='pt')
                # PREDICT MISSING WORDS
                pred = self.model(indices)
                masked_indices = torch.where(indices == self.mask_token_id)[1]
                # TOP 10 PREDICTIONS
                top10 = torch.topk(pred[0][0][masked_indices, :], top_k, axis=1)
                # FILL IN MISSING WORDS
                for i in range(top_k):
                    temp = text
                    for j in range(self.mask_num):
                        if j < top10.indices.shape[0] and i < top10.indices.shape[1]:
                            temp = temp.replace(self.mask_token, self.tokenizer.decode(top10.indices[j][i]), 1)
                    temp = re.sub(r'\s+', ' ', temp)
                    new_text.append(temp)
        return new_text

In [11]:
data.columns

Index(['Unnamed: 0', 'image_id', 'caption', 'funny_score_x', 'caption_id',
       'funny_score_y', 'fc_mean', 'generated'],
      dtype='object')

In [12]:
data

,Unnamed: 0,image_id,caption,funny_score_x,caption_id,funny_score_y,fc_mean,generated
0,5946,bokete_654,wanted criminals whose crimes are too detailed...,0.000000,caption5945,0.0,29.333333,0
1,6038,bokete_654,"my car license, it won't fit in my pocket.",0.000010,caption6037,1.0,29.333333,0
2,6154,bokete_654,full name,0.000886,caption6153,87.0,29.333333,0
3,21791,bokete_2176,"lowercase ""y"".",0.007244,caption21785,711.0,69.727273,0
4,21850,bokete_2206,"""oh, that's mr. suzuki's husband.",0.000010,caption21844,1.0,25.857143,0
...,...,...,...,...,...,...,...,...
1484177,3261661,imgflip_1824,when your dnd character doesn't match the setting,0.000000,caption3232465,0.0,36.322581,0
1484178,3261662,imgflip_1824,if you are riding a bike while high on shrooms...,0.000000,caption3232466,0.0,36.322581,0
1484179,3261663,imgflip_1824,"&quot;vader, you can't go outside with those h...",0.000000,caption3232467,0.0,36.322581,0
1484180,3261664,imgflip_1824,vader in his downtime,0.000000,caption3232468,0.0,36.322581,0


In [13]:
data.drop(columns=['Unnamed: 0','caption_id'], inplace=True)

In [14]:
data = data.sort_values(by=['image_id', 'funny_score_y'], ascending=[True, False])
data

,image_id,caption,funny_score_x,funny_score_y,fc_mean,generated
6651,bokete_100219,listen to me. listen to me. it's a relief that...,1.000000,98148.0,1637.000000,0
6676,bokete_100219,"that's not what ""use your feet"" means.",0.000306,30.0,1637.000000,0
6725,bokete_100219,"""you know, don't give up your asthma every tim...",0.000082,8.0,1637.000000,0
6663,bokete_100219,"hi, i'm your rival's father.",0.000051,5.0,1637.000000,0
6681,bokete_100219,"""why are you wearing that?""",0.000041,4.0,1637.000000,0
...,...,...,...,...,...,...
1473644,imgflip_996,after you and your friend bids goodnight after...,0.000010,1.0,44.690909,0
1473645,imgflip_996,my balls; the rats in the kfc deep fryer,0.000010,1.0,44.690909,0
1473646,imgflip_996,what are you doing here? just looking for actu...,0.000000,0.0,44.690909,0
1473647,imgflip_996,"me playing super doomspire; captainspinxs, por...",0.000000,0.0,44.690909,0


In [15]:
old = pd.read_csv('../Data/Oxford_HIC/Generate_bert_top1000_oxford.csv')
new = pd.read_csv('../Data/Oxford_HIC/Generate_bert_top1000_oxford_1.csv')
mix = pd.concat([old, new], ignore_index=True)
mix = mix.drop_duplicates()
mix.to_csv('../Data/Oxford_HIC/Generate_bert_top1000_oxford_mix.csv')

In [ ]:
import os
if os.path.exists('../Data/Oxford_HIC/Generate_bert_top1000_oxford_mix.csv'):
    new_data = pd.read_csv('../Data/Oxford_HIC/Generate_bert_top1000_oxford_mix.csv')
    for i in range(data.shape[0]):
        if data.image_id[i] in new_data.image_id.values:
            save = (new_data.shape[0] // 10000)+1
            save_id = i
        else:
            break
else:
    save = 0
    save_id = 0
print(f'save: {save}, save_id: {save_id}')
new_data = pd.DataFrame(columns=['image_id', 'caption', 'funny_score_x', 'funny_score_y', 'fc_mean', 'generated'])
present = pd.DataFrame(columns=['image_id', 'caption', 'funny_score_x', 'funny_score_y', 'fc_mean', 'generated'])
skip = None

with tqdm(total=data.shape[0]) as progress_bar:
    for i in range(data.shape[0]):
        if i <= save_id:
            progress_bar.update()
            continue
        progress_bar.update()
        if skip != None:
            if data.caption[i] == skip:
                progress_bar.set_postfix({"j": j, "k": len(temp_sentences),"img":data.image_id[i], "img_size": present.shape, "size": new_data.shape, "saveID": save_id})
                continue
            else:
                skip = None
        if new_data.shape[0] // 10000 > save:
            save = (new_data.shape[0] // 10000)+1
            save_id = i
            new_data.to_csv('../Data/Oxford_HIC/Generate_bert_top1000_oxford_2.csv', index=False)
        temp = data[data['image_id'] == data['image_id'][i]]
        if present.empty == False:
            temp = pd.concat([temp, present], ignore_index=True)
            temp = temp.drop_duplicates()
            if present.image_id[0] == data.image_id[i]:
                if temp.shape[0] > 800:
                    new_data = pd.concat([new_data, temp], ignore_index=True)
                    new_data = new_data.drop_duplicates()
                    present = pd.DataFrame(columns=['image_id', 'caption', 'funny_score_x', 'funny_score_y', 'fc_mean', 'generated'])
                    skip = data.caption[i]
                    progress_bar.set_postfix({"j": j, "k": len(temp_sentences),"img":data.image_id[i], "img_size": present.shape, "size": new_data.shape, "saveID": save_id})
                    continue
            else:
                new_data = pd.concat([new_data, temp], ignore_index=True)
                present = pd.DataFrame(columns=['image_id', 'caption', 'funny_score_x', 'funny_score_y', 'fc_mean', 'generated'])
                progress_bar.set_postfix({"j": j, "k": len(temp_sentences),"img":data.image_id[i], "img_size": present.shape, "size": new_data.shape, "saveID": save_id})
        text = data.caption[i]
        image_id = data.image_id[i]
        funnyscore_x = data.funny_score_x[i]
        funnyscore_y = data.funny_score_y[i]
        fc_mean = data.fc_mean[i]
        generated = data.generated[i]
        generate_wordcount = len(nltk.word_tokenize(text)) - 3
        # print(f'image_id{image_id}, funnyscore{funnyscore}, generate_wordcount{generate_wordcount}, text{text}')
        if generate_wordcount > 0:
            for j in range(min(generate_wordcount, 5)):
                # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
                lm = LMmask(mask_num = j+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
                temp_sentences = lm.apply(text, top_k=5)

                for sentence in temp_sentences:
                    if present.empty:
                        present = pd.DataFrame([[image_id, sentence, funnyscore_x, funnyscore_y, fc_mean, 1]], columns=['image_id', 'caption', 'funny_score_x', 'funny_score_y', 'fc_mean', 'generated'])
                    else:
                        present = pd.concat([present, pd.DataFrame([[image_id, sentence, funnyscore_x, funnyscore_y, fc_mean, 1]], columns=['image_id', 'caption', 'funny_score_x', 'funny_score_y', 'fc_mean', 'generated'])], ignore_index=True)
                progress_bar.set_postfix({"j": j, "k": len(temp_sentences),"img":data.image_id[i], "img_size": present.shape, "size": new_data.shape, "saveID": save_id})
new_data['caption'] = new_data['caption'].str.lower()
# remove duplicate data
new_data = new_data.drop_duplicates()

save: 43, save_id: 2227


  0%|          | 2234/1484182 [00:29<8:10:43, 50.33it/s, j=4, k=55, img=bokete_62571, img_size=(1290, 6), size=(0, 6), saveID=2227] C:\Users\TonyLab\AppData\Local\Temp\ipykernel_14184\868929101.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, temp], ignore_index=True)
  0%|          | 2592/1484182 [30:26<2510:49:57,  6.10s/it, j=4, k=45, img=bokete_78963, img_size=(0, 6), size=(61884, 6), saveID=2227]     

In [202]:
new_data

,image_id,caption,funny_score_x,funny_score_y,fc_mean,generated,caption_id
0,bokete_2176,"lowercase ""y"".",0.007244,711.0,69.727273,0,caption21785
1,bokete_2176,"anpanman, new face!",0.000132,13.0,69.727273,0,caption23751
2,bokete_2176,it's a driving shot!,0.000122,12.0,69.727273,0,caption22920
3,bokete_2176,"if it's not moving, it's going to leak ...",0.000061,6.0,69.727273,0,caption23818
4,bokete_2176,fracture of left ankle,0.000061,6.0,69.727273,0,caption97059
...,...,...,...,...,...,...,...
912,bokete_2232,"hey, if i were to sell it at an online auction...",0.000010,1.0,30.952381,0,caption26965
913,bokete_2232,is this one of the nuts and this one of janet?,0.000010,1.0,30.952381,0,caption47684
914,bokete_2232,do you want me to race for chicken?,0.000010,1.0,30.952381,0,caption385434
915,bokete_2232,but~~~ you know~~ two more cars are going to b...,0.000000,0.0,30.952381,0,caption22476


In [188]:
image_id_counts = new_data['caption'].value_counts()
image_id_counts

caption
bokete_654    271
Name: count, dtype: int64

In [ ]:
# 計算每個 image_id 的資料數量
image_id_counts = new_data['image_id'].value_counts()
print(f'Number of unique image_id: {len(image_id_counts)}')
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 800].index
print(f'Number of image_id with 300 captions: {len(valid_image_ids)}')
# 篩選原始資料
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print(f'Number of data: {filtered_data.shape[0]}')
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(300)
)
print(f'Number of data: {top_captions.shape[0]}')

In [ ]:
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# mcdonalds images: 9/306 , data = 2700/3654/29804
# mcdonalds_switzerland images: 262/1125 , data = 78600/110815/226667
# mcdonaldscanada images: 150/843 , data = 45000/86156/179124
# sonicdrivein images: 447/2208 , data = 134100/222368/460574
# wendys images: 14/367 , data = 4200/5846/35727
print(new_data.shape)
new_data.to_csv('../Data/Instagram/Generate_parrot_bert_sonicdrivein.csv', index=False)

In [16]:
import pandas as pd
print("shape of data: ", data.shape)
print()
image_id_counts = mix['image_id'].value_counts()
######################################################################################################
print(f'                  Oxford_HIC')
print(f'           Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 300]
# sum = len(x)
# print(f' {300} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 200]
# x = x[x < 300]
# sum += len(x)
# print(f'{200:4d} <= caption < {300:4d} --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 100]
# x = x[x < 200]
# sum += len(x)
# print(f'{100:4d} <= caption < {200:4d} --- {len(x):6d} --- {sum:6d}')
for i in range(10):
    x = image_id_counts[image_id_counts >= (10-1-i)*100]
    x = x[x < (10-i)*100]
    sum += len(x)
    print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')

shape of data:  (1484182, 6)

                  Oxford_HIC
           Total image counts: 196
 1000 <= caption        ---    136 ---    136
  900 <= caption < 1000 ---     10 ---    146
  800 <= caption <  900 ---      5 ---    151
  700 <= caption <  800 ---      3 ---    154
  600 <= caption <  700 ---      6 ---    160
  500 <= caption <  600 ---     10 ---    170
  400 <= caption <  500 ---      7 ---    177
  300 <= caption <  400 ---      4 ---    181
  200 <= caption <  300 ---      6 ---    187
  100 <= caption <  200 ---      3 ---    190
    0 <= caption <  100 ---      6 ---    196


# Generate data not in dictionary

In [ ]:
from string import punctuation
import pandas as pd
from sklearn.model_selection import train_test_split

# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/CaptionID_wendys.csv'

data = pd.read_csv(dirPath)

In [ ]:
# lower caption
data['caption'] = data['caption'].str.lower()

In [ ]:
import pandas as pd
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
import albumentations
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

class NLPTransform(BasicTransform):
    """ Transform for nlp task."""

    @property
    def targets(self):
        return {"data": self.apply}

    def update_params(self, params, **kwargs):
        if hasattr(self, "interpolation"):
            params["interpolation"] = self.interpolation
        if hasattr(self, "fill_value"):
            params["fill_value"] = self.fill_value
        return params

    def get_sentences(self, text, lang='en'):
        return sent_tokenize(text)

In [ ]:
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
from transformers import BertTokenizer, AutoTokenizer, RobertaTokenizer,BertForMaskedLM, RobertaForMaskedLM, AutoModelForMaskedLM
import albumentations
import torch
import re
import string

class LMmask(NLPTransform):

    def __init__(self, mask_num = 1, tokenizer_name='bert-base-uncased', model_name='bert-base-uncased'):
        self.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~…“”–’'
        self.mask_num = mask_num
        if tokenizer_name == 'BertTokenizer':
            #   BertTokenizer + BertForMaskedLM  ==> 'bert-base-uncased'
            self.tokenizer = BertTokenizer.from_pretrained(model_name)
            self.model = BertForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 103
        elif tokenizer_name == 'RobertaTokenizer':
            #   RobertaTokenizer + RobertaForMaskedLM  ==> 'FacebookAI/roberta-base'
            self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
            self.model = RobertaForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 50264
        elif tokenizer_name == 'AutoTokenizer':
            # AlbertConfig, BartConfig, BertConfig, BigBirdConfig, CamembertConfig, ConvBertConfig, Data2VecTextConfig, DebertaConfig, DebertaV2Config, DistilBertConfig, ElectraConfig, ErnieConfig, EsmConfig, FlaubertConfig, FNetConfig, FunnelConfig, IBertConfig, LayoutLMConfig, LongformerConfig, LukeConfig, MBartConfig, MegaConfig, MegatronBertConfig, MobileBertConfig, MPNetConfig, MraConfig, MvpConfig, NezhaConfig, NystromformerConfig, PerceiverConfig, QDQBertConfig, ReformerConfig, RemBertConfig, RobertaConfig, RobertaPreLayerNormConfig, RoCBertConfig, RoFormerConfig, SqueezeBertConfig, TapasConfig, Wav2Vec2Config, XLMConfig, XLMRobertaConfig, XLMRobertaXLConfig, XmodConfig, YosoConfig.
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)


            self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def apply(self, data, top_k=10):
        new_text = []
        split = data.split()
        def generateINDICT(split, idx, former):
            n = split[idx]
            token = self.tokenizer.encode(n, return_tensors="pt")
            single = token.shape[1] == 3

            if not single:
                words = data.replace(n, self.mask_token, 1)
                # DEFINE SENTENCE
                indices = self.tokenizer.encode(words, add_special_tokens=True, return_tensors='pt')
                # PREDICT MISSING WORDS
                pred = self.model(indices)
                masked_indices = torch.where(indices == self.mask_token_id)[1]
                # TOP 10 PREDICTIONS
                top10 = torch.topk(pred[0][0][masked_indices, :], top_k, axis=1)
                # FILL IN MISSING WORDS
                for i in range(top_k):
                    temp = words
                    for j in range(self.mask_num):
                        if j < top10.indices.shape[0] and i < top10.indices.shape[1]:
                            temp = temp.replace(self.mask_token, self.tokenizer.decode(top10.indices[j][i]), 1)
                    temp = re.sub(r'\s+', ' ', temp)
                    if idx + 1 < len(split):
                        generateINDICT(split, idx + 1, former)
                    else:
                        new_text.append(temp)
            else:
                if idx + 1 < len(split):
                    generateINDICT(split, idx + 1, former)
                else:
                    new_text.append(former)
        generateINDICT(split, 0, data)
        return new_text

In [ ]:
words_not_in_dict = []
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
for i in range(data.shape[0]):
    text = data.caption[i]
    # remove punctuation
    text = text.translate(str.maketrans('', '', '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~…“”–’'))
    #lowercase
    text = text.lower()
    split = text.split()
    for word in split:
        single = tokenizer.encode(word, return_tensors="pt").shape[1] == 3
        # check if word is number
        number = False
        try:
            int(word)
            number = True
        except:
            pass
        if not single and not number:
            words_not_in_dict.append(word)

In [ ]:
words_not_in_dict = set(words_not_in_dict)
print(len(words_not_in_dict))
words_not_in_dict

In [ ]:
temp = tokenizer.encode('lol', return_tensors="pt")
for i in range(temp.shape[1]):
    print(f'{i}: {tokenizer.decode(temp[0][i])}')
# tokenizer.decode(101)

In [ ]:
# complete following code
new_data = pd.DataFrame(columns=['caption', 'image_id', 'funny_score'])

with tqdm(total=data.shape[0]) as progress_bar:
    for i in range(data.shape[0]):
        text = data.caption[i]
        image_id = data.image_id[i]
        funnyscore = data.funny_score[i]
        new_data = pd.concat([new_data, pd.DataFrame([[text, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
        generate_wordcount = len(nltk.word_tokenize(text)) - 3
        # print(f'image_id{image_id}, funnyscore{funnyscore}, generate_wordcount{generate_wordcount}, text{text}')
        if generate_wordcount > 0:
            for j in range(min(generate_wordcount, 1)):
                # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
                lm = LMmask(mask_num = j+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
                temp_sentences = lm.apply(text, top_k=5)

                for sentence in temp_sentences:
                    new_data = pd.concat([new_data, pd.DataFrame([[sentence, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
                progress_bar.set_postfix({"j": j, "k": len(temp_sentences), "size": new_data.shape})
        progress_bar.update()
new_data['caption'] = new_data['caption'].str.lower()
# remove duplicate data
new_data = new_data.drop_duplicates()

In [ ]:
new_data

In [ ]:
# 計算每個 image_id 的資料數量
image_id_counts = new_data['image_id'].value_counts()
print(f'Number of unique image_id: {len(image_id_counts)}')
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 300].index
print(f'Number of image_id with 300 captions: {len(valid_image_ids)}')
# 篩選原始資料
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print(f'Number of data: {filtered_data.shape[0]}')
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(300)
)
print(f'Number of data: {top_captions.shape[0]}')

In [ ]:
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# mcdonalds_switzerland images: 262/1125 , data = 78600/110815/226667
print(new_data.shape)
new_data.to_csv('../Data/Instagram/Generate_mcdonalds_switzerland.csv', index=False)# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# mcdonalds_switzerland images: 262/1125 , data = 78600/110815/226667
print(new_data.shape)
new_data.to_csv('../Data/Instagram/Generate_mcdonalds_switzerland.csv', index=False)

# Check distribution of how many captions in each image

In [ ]:
import pandas as pd
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
name = 'wendys'
fileName = 'Generate_' + name
dirPath = '../Data/Instagram/' + fileName + '.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
print()
image_id_counts = data['image_id'].value_counts()
######################################################################################################
print(f'          {fileName}')
print(f'           Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 300]
# sum = len(x)
# print(f' {300} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 200]
# x = x[x < 300]
# sum += len(x)
# print(f'{200:4d} <= caption < {300:4d} --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 100]
# x = x[x < 200]
# sum += len(x)
# print(f'{100:4d} <= caption < {200:4d} --- {len(x):6d} --- {sum:6d}')
for i in range(10):
    x = image_id_counts[image_id_counts >= (10-1-i)*100]
    x = x[x < (10-i)*100]
    sum += len(x)
    print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


#########################################

In [ ]:
text = "You can’t achieve strawberry lemonade until you first get strawberry lemonade followed by strawberry lemonade."
new_data = pd.DataFrame()
for i in range(3):
    # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
    lm = LMmask(mask_num = i+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
# lm = LMmask(mask_num = 1, tokenizer_name='AutoTokenizer', model_name='')
    # 'DataFrame' object has no attribute 'append'

    new_data = pd.concat([new_data, pd.DataFrame(lm.apply(text, top_k=5))], ignore_index=True)
    new_data = pd.concat([new_data, pd.DataFrame(["============================================================="])], ignore_index=True)
new_data
# lm = LMmask(mask_num = 1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
# lm.apply(text, top_k=5)


In [ ]:
new_data.to_csv('./roberta2.csv', index=False)

In [ ]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
tokenizer.decode(2226)


In [ ]:
import string
from torch.utils.data import Dataset, DataLoader
import clip
import os
import pandas as pd
import pickle
from torch import nn
import numpy as np
import torch
import torch.nn.functional as nnf
import sys
from typing import Tuple, List, Union, Optional
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    AdamW,
    get_linear_schedule_with_warmup,
)
from transformers import AutoConfig, AutoTokenizer, Gemma2ForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
import PIL.Image
from tqdm import tqdm
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
inwordList = set()
outwordList = set()
progress_counter = tqdm(total=len(data), desc='Counting tokens', position=0, leave=True)
progress_inDict = tqdm(total=len(data), desc='Counting in dict', position=0, leave=True)
def token_counter(caption):
    caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    words = caption.split()
    temp = set(words)
    progress_counter.update(1)
    return len(temp)

def word_in_dict(caption):
    # caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    words = caption.split()
    temp = set(words)
    notTested = temp.difference(inwordList).difference(outwordList)
    notInDictCounter = len(temp.intersection(outwordList))
    # print(f'notInDictCounter: {notInDictCounter}, notTested: {notTested}')
    for word in notTested:
        token = tokenizer(word, return_tensors="pt").input_ids
        # print(f'word: {word}, token: {token}')
        if token.shape[1] == 1:
            inwordList.add(word)
        else:
            notInDictCounter += 1
            outwordList.add(word)
    # progress_inDict.update(1)
    return notInDictCounter

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# 計算每個 caption 的字數
data['token_count'] = data['caption'].apply(token_counter)
progress_counter.close()
# 計算每個 caption 中不在字典內的字數
data['out_of_dict_count'] = data['caption'].apply(word_in_dict)
progress_inDict.close()
print(f'Number of unique words in the dictionary: {len(inwordList)}')
print(f'Number of unique words out of the dictionary: {len(outwordList)}')

In [ ]:
test_tokens = inwordList.union(outwordList)

In [ ]:
train_tokens = inwordList.union(outwordList)

In [ ]:
a = set(test_tokens)
b = set(train_tokens)
c = set()
d = set()
for testDatasetToken in a:
    c.add(testDatasetToken.item())
print(len(c))
for testDatasetToken in b:
    d.add(testDatasetToken.item())
print(len(d))
print(len(c.symmetric_difference(d)))
print(len(c.intersection(d)))

In [ ]:
print(data.shape)
print(data[data['out_of_dict_count'] < 4].shape)
print(data[data['out_of_dict_count'] < 3].shape)
print(data[data['out_of_dict_count'] < 2].shape)
print(data[data['out_of_dict_count'] < 1].shape)
print(data[data['out_of_dict_count'] < 0].shape)
data.describe()

In [ ]:
data = data[data['out_of_dict_count'] < 1]

In [ ]:
# 計算每個 image_id 的資料數量
image_id_counts = data['image_id'].value_counts()
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 1000].index
# 篩選原始資料
filtered_data = data[data['image_id'].isin(valid_image_ids)]
print(len(valid_image_ids))
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1000)
)
print(top_captions.shape)
print(filtered_data.shape)

In [ ]:
print(f'          Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
for i in range(10):

    if i == 9:
        x = image_id_counts[image_id_counts >= 50]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {50:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')
        x = image_id_counts[image_id_counts >= 10]
        x = x[x < 50]
        sum += len(x)
        print(f' {10:4d} <= caption < {50:4d} --- {len(x):6d} --- {sum:6d}')
        x = image_id_counts[image_id_counts < 10]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {10:4d} --- {len(x):6d} --- {sum:6d}')
    else:
        x = image_id_counts[image_id_counts >= (10-1-i)*100]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


In [ ]:
# get data differ from data_all and data1500
print(data.shape)
print(filtered_data.shape)
exceptdata = data[~data['image_id'].isin(filtered_data['image_id'])]
print(len(exceptdata['image_id'].value_counts()))
print(exceptdata.shape)
exceptdata.to_csv('../Data/Oxford_HIC/except1000up_oxford_hic_data.csv', index=False)

In [ ]:
filtered_data.to_csv('../Data/Oxford_HIC/1000up_oxford_hic_data.csv', index=False)

In [ ]:
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1)
)
top_captions.shape
top_captions.to_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv', index=False)

In [ ]:
top_captions.to_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv', index=False)

In [ ]:
whole = pd.read_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv')
unique_image_ids = whole['image_id'].unique()

train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
train = whole[whole['image_id'].isin(train_ids)]
test = whole[whole['image_id'].isin(test_ids)]
print(train.shape, test.shape)

In [ ]:
train_mess = pd.DataFrame()
test_mess = pd.DataFrame()
for image_id, group in whole.groupby("image_id"):
    train_split, test_split = train_test_split(group, test_size=0.2, random_state=42)
    train_mess = pd.concat([train_mess, train_split])
    test_mess = pd.concat([test_mess, test_split])
print(f'train: {train_mess.shape}')
print(f'test: {test_mess.shape}')

In [ ]:
train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504']
train_data = train[train['image_id'].isin(train_image)]
test_image = ['bokete_100174', 'imgflip_834']
test_data = test[test['image_id'].isin(test_image)]
print(train_data.shape, test_data.shape)
train_data.to_csv('../Data/Oxford_HIC/Train_Only10_oxford_hic_data.csv', index=False)
test_data.to_csv('../Data/Oxford_HIC/Test_Only10_oxford_hic_data.csv', index=False)

In [ ]:
train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504','bokete_100174', 'imgflip_834']
train_data_mess = train_mess[train_mess['image_id'].isin(train_image)]
test_image =  ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504','bokete_100174', 'imgflip_834']
test_data_mess = test_mess[test_mess['image_id'].isin(test_image)]
print(train_data_mess.shape, test_data_mess.shape)
train_data.to_csv('../Data/Oxford_HIC/Train_Only10_mess_oxford_hic_data.csv', index=False)
test_data.to_csv('../Data/Oxford_HIC/Test_Only10_mess_oxford_hic_data.csv', index=False)

In [ ]:
train_data_mess

In [ ]:
# list same caption in train_data and test_data_mess
a = test_data[test_data['image_id'] == 'imgflip_130']
b = test_data_mess[test_data_mess['image_id'] == 'imgflip_130']
x = set(a['caption']).intersection(set(b['caption']))
for i in x:
    if '' in i:
        print(i)
# 2spbgym,climb a mountain? pff, i have wings...
# all-the-things,go to a pizza buffet eat all the pizza
# imgflip_0,12 dollars; 11 dollars with 1 dollar shipping
# imgflip_1033,I POUR MILK BEFORE CEREAL
# imgflip_11,ME WAITING FOR MY INTERNET TO RECONNECT
# imgflip_117,calling the teacher mom
# imgflip_16,IF SOMEONE DIES IN THE LIVING ROOM... IS IT STILL CALLED THE LIVING ROOM?
# imgflip_189,you; losing a few seconds of your life looking at this
# imgflip_23,me: gets up and starts clapping because the chiefs won; the guy who has been pushing my wheelchair for 10 years
# imgflip_504,THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS

# i-love-coloring-kid,she started writing notes !!
# imgflip_130,WHEN YOUR FRIEND; DOSENT LIKE ROOT BEER

In [ ]:
a

In [ ]:
img_names = data['image_id'].unique()
len(img_names)

In [ ]:
top_captions = (
    data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(50)
)
top_captions.shape

In [ ]:
train_data = pd.DataFrame()
test_data = pd.DataFrame()
only = pd.DataFrame()
for image_id, group in top_captions.groupby("image_id"):
    if group.shape[0] < 50:
        continue
    only = pd.concat([only, group])
    # train, test = train_test_split(group, test_size=0.2, random_state=42)
    # train_data = pd.concat([train_data, train])
    # test_data = pd.concat([test_data, test])
print(only.shape)

In [ ]:
only.to_csv('../Data/Oxford_HIC/Only50_oxford_hic_data.csv', index=False)

In [ ]:
top_captions.to_csv('../Data/Oxford_HIC/Top10_oxford_hic_data.csv', index=False)

In [ ]:
# 獲取唯一的 image_id
unique_image_ids = top_captions['image_id'].unique()
print(unique_image_ids.shape)
# 將 image_id 拆分為 80% 訓練集和 20% 測試集
train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)

# 根據拆分的 image_id 選取資料
train_data = top_captions[top_captions['image_id'].isin(train_ids)]
test_data = top_captions[top_captions['image_id'].isin(test_ids)]
print(train_data.shape, test_data.shape)

In [ ]:
unique_image_ids.shape[0]/3

In [ ]:
a = unique_image_ids[:30000]
print(a.shape)

In [ ]:
dirPath = '../Data/Oxford_HIC/Only1200_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
######################################################################################################
train = pd.DataFrame()
test = pd.DataFrame()
for image_id, group in data.groupby("image_id"):
    train_split, test_split = train_test_split(group, test_size=0.2, random_state=42)
    train = pd.concat([train, train_split])
    test = pd.concat([test, test_split])
print(f'train: {train.shape}')
print(f'test: {test.shape}')
######################################################################################################
# unique_image_ids = data['image_id'].unique()
# # unique_image_ids = unique_image_ids[:30000]
# # unique_image_ids, rest = train_test_split(unique_image_ids, test_size=0.7, random_state=42)
# # print(unique_image_ids.shape)
# train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
# train = data[data['image_id'].isin(train_ids)]
# test = data[data['image_id'].isin(test_ids)]
# print(train.shape, test.shape)

In [ ]:
train=train.reset_index()
test=test.reset_index()

In [ ]:



train_image = ['imgflip_0', 'imgflip_101', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_47', 'imgflip_504']
train_text = ['12 dollars; 11 dollars with 1 dollar shipping'
              ,'I HAD A GIRLFRIEND; AAAAAAND ITS GONE'
              ,'I POUR THE CEREAL AFTER I POUR THE MILK'
              ,'WAITING FOR MY PHONE TO GET  TO 100%'
              ,'You when you have over one test at school in a day'
              ,'IF SOMEONE WANTS TO KILL YOU; GO TO A LIVING ROOM'
              ,'you; eating 5 pounds of cheese; every day; your stomach'
              ,'Me:stands up to stretch my legs; The person who had been pushing my wheelchair for the last 26 years'
              ,'Me: Opens door for some fresh air; Everyone else in the submarine:'
              ,'THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS']
# imgflip_130,WHEN YOU SEE PICS OF YOUR FRIENDS HANGING OUT; BUT YOU WEREN'T INVITED
# imgflip_659,When the teacher uses your voice recording on the homework as an example
test_image = ['imgflip_130', 'imgflip_659']
test_text = ['0 VIEWS 5 DISLIKES'
              ,'when the mobile game ad is so laggy that it crashes your game and you lose out on a reward:']

tokens_list = []
mask_list = []
prefix_list = []
train_gt = []
train_caption = dict()
train_image_id_list = []
# mess = set()
mess_a = set()
mess_b =set()
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
print(train.shape)
for i in range(train.shape[0]):
    caption = train['caption'][i]
    image_id = train['image_id'][i]
    # if image_id == 'imgflip_117':#and 'GONE' in caption:
    #     notmess.add(caption)
    #     print(f"Image ID: {image_id}, Caption: {caption}")
# train_image = ['', 'imgflip_101', '','', 'imgflip_117', '', '','', '', '']


    if image_id in train_image and caption in train_text:
        print(f"Image ID: {image_id}, Caption: {caption}")
print("===================================================================================")
for i in range(test.shape[0]):
    caption = test['caption'][i]
    image_id = test['image_id'][i]
    if image_id == 'imgflip_130':#and 'GONE' in caption:
        mess_a.add(caption)
    if image_id == 'imgflip_659':
        mess_b.add(caption)
    if image_id in test_image and caption in test_text:
        print(f"Image ID: {image_id}, Caption: {caption}")


In [ ]:
set.intersection(notmess_b, mess_b)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-2.7B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-2.7B")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")